In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NguyenAn05/pocket-r1.git'
REPO_DIR = '/tmp/pocket-r1'
INIT_LORA_DIR = '/kaggle/input/pocket-r1-exp1b-lora'
PRM_DIR = '/kaggle/input/pocket-r1-deberta-prm-v1'
OUT_LORA_DIR = '/kaggle/working/exp4_grpo_hybrid_lora'
MERGED_DIR = '/tmp/exp4_grpo_hybrid_merged'
OUT_REPORT_DIR = '/kaggle/working/exp4_grpo_hybrid'
os.makedirs(OUT_REPORT_DIR, exist_ok=True)

print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], text=True))
print('Init LoRA files:', os.listdir(INIT_LORA_DIR))
print('DeBERTa PRM files:', os.listdir(PRM_DIR))

if os.path.exists(REPO_DIR):
    subprocess.check_call(['rm', '-rf', REPO_DIR])
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'bitsandbytes', 'accelerate', 'sentencepiece', 'vllm'])
print('Phase 4 Dual-GPU GRPO environment ready!')


In [ ]:
import subprocess, sys

# Dual-GPU GRPO: GPU 0 = Qwen3-4B QLoRA Actor (from Exp 1b), GPU 1 = DeBERTa-v3 PRM v1 (FP16)
subprocess.check_call([
    sys.executable, f'{REPO_DIR}/scripts/train_grpo.py',
    '--config', f'{REPO_DIR}/configs/rl/exp4_grpo_hybrid.yaml',
    '--init-lora-dir', INIT_LORA_DIR,
    '--prm-model-dir', PRM_DIR,
    '--output-dir', OUT_LORA_DIR,
    '--merged-dir', MERGED_DIR,
    '--num-steps', '100',
    '--group-size', '4',
    '--merge',
], cwd=REPO_DIR)
print('Exp 4 GRPO Hybrid training & LoRA merge complete!')


In [ ]:
import subprocess, time, urllib.request, sys, os

LOG_PATH = '/tmp/vllm_exp4.log'
log_file = open(LOG_PATH, 'w')

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MERGED_DIR,
    '--served-model-name', 'Qwen/Qwen3-4B-exp4-grpo-hybrid',
    '--dtype', 'half',
    '--tensor-parallel-size', '2',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--host', '127.0.0.1',
    '--port', '8000',
]
print('Starting vLLM for Exp 4:', ' '.join(vllm_cmd))
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

ready = False
for sec in range(420):
    if vllm_proc.poll() is not None:
        log_file.flush()
        with open(LOG_PATH) as f:
            print('vLLM exited early! Last logs:\n', f.read()[-4000:])
        raise RuntimeError(f'vLLM crashed with code {vllm_proc.returncode}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                print(f'vLLM ready in {sec}s:', r.read().decode())
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    vllm_proc.terminate()
    raise TimeoutError('vLLM did not become ready within 420s')

try:
    eval_cmd = [
        sys.executable, f'{REPO_DIR}/scripts/run_exp0_baseline.py',
        '--config', f'{REPO_DIR}/configs/eval/exp0_baseline.yaml',
        '--backend', 'openai_compatible',
        '--model', 'Qwen/Qwen3-4B-exp4-grpo-hybrid',
        '--base-url', 'http://127.0.0.1:8000/v1',
        '--workers', '24',
        '--no-ev',
        '--experiment-name', 'exp4_grpo_hybrid',
        '--output-dir', OUT_REPORT_DIR,
    ]
    print('Evaluating Exp 4 on 2000 Held-out hands:', ' '.join(eval_cmd))
    subprocess.check_call(eval_cmd, cwd=REPO_DIR)
finally:
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=20)
    except Exception:
        vllm_proc.kill()
    log_file.close()
    print('Exp 4 complete! Outputs:', os.listdir(OUT_REPORT_DIR))
